In [0]:
%pip install gspread google-auth
dbutils.library.restartPython()

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict,
    scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

SHEET_ID = "1uiv27J0YPDWqSVuN-QLpS-fOcfxAjewK5ajTd3aA8vI"
sh = gc.open_by_key(SHEET_ID)

print("Connected to:", sh.title)

In [0]:
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))

df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

print("Row count:", df.count())
df.printSchema()

In [0]:
bounds_row = spark.sql("""
  SELECT
    timestamp(floor(unix_timestamp(current_timestamp()) / 900) * 900 - 900)       AS window_end_utc,
    timestamp(floor(unix_timestamp(current_timestamp()) / 900) * 900 - 1800) AS window_start_utc
""").collect()[0]

window_start_utc = bounds_row['window_start_utc']
window_end_utc   = bounds_row['window_end_utc']

meta_row = spark.sql(f"""
  SELECT
    MOD(FLOOR(DATEDIFF(to_date(from_utc_timestamp(timestamp('{window_start_utc}'),'Europe/London')), DATE'2025-12-14')/7)+46,52) AS Week,
    concat(
      date_format(from_utc_timestamp(timestamp('{window_start_utc}'),'Europe/London'),'dd/MM/yyyy HH:mm'), ' - ',
      date_format(from_utc_timestamp(timestamp('{window_end_utc}'),'Europe/London'),'dd/MM/yyyy HH:mm')
    ) AS date_time_range
""").collect()[0]

week_val = meta_row['Week']
date_time_range = meta_row['date_time_range']

print(f"Window: {window_start_utc} - {window_end_utc}")
print(f"Week: {week_val} | Range: {date_time_range}")

In [0]:
reports = [
    {"tab": "Data", "name": "D.Analysis - OSR PiE",
     "area": "('Pick','Pie Station')", "event": "('COUN','MSKU','PSKU','RSIN')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - OSR Topup",
     "area": "('Topup','TopUp','PiOrQi')", "event": "('QSIN','TARS','TPUT')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - E3 Packing",
     "area": "('E3 - Packing')", "event": "('PackingParcelCompleteEvent','PackingItemScannedEvent')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - Parcel Induct",
     "area": "('Parcel Induct')", "event": "('APAR','PIND','SPAR','UPAR')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - Parcel Sortation",
     "area": None, "event": "('ParcelSortedToSack','SackMappedToPosition','SackUnMappedFromPosition')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - Inbound Decanting",
     "area": "('Inbound Decanting')", "event": "('DECN','TOPR')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - OSR Decanting",
     "area": "('OSR Decanting')", "event": "('ODEC','OTOP')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - BCR Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%RET%')"},

    {"tab": "Data", "name": "D.Analysis - E1/E2 Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND NOT EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%RET%')"},
]

COLUMNS = ['Date', 'Hour', 'PAYLOAD_BONUSCODE', 'PAYLOAD_EVENTTYPE',
           'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
           'Week', 'Date Time Range', 'Report Name']

In [0]:
try:
    ws = sh.worksheet("Data")
except gspread.exceptions.WorksheetNotFound:
    ws = sh.add_worksheet(title="Data", rows=2000, cols=len(COLUMNS))
    ws.update('A1', [COLUMNS])
    print("Created 'Data' tab with header")

In [0]:
import pandas as pd

failures = []
total_rows_pushed = 0

for r in reports:
    try:
        area_clause = f"AND PAYLOAD_AREACODE IN {r['area']}" if r['area'] else ""

        query = f"""
          SELECT
            date_format(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London'),'dd/MM/yyyy') AS Date,
            hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London')) AS Hour,
            PAYLOAD_BONUSCODE,
            PAYLOAD_EVENTTYPE,
            SUM(PAYLOAD_QUANTITY)      AS Total_Quantity,
            SUM(PAYLOAD_STANDARDHOURS) AS Total_StandardHours,
            SUM(PAYLOAD_SMV)           AS Total_SMV
          FROM landing_bonus_hub_event_parsed
          WHERE TRIM(PAYLOAD_WAREHOUSECODE) = 'X'
            {area_clause}
            AND PAYLOAD_EVENTTYPE IN {r['event']}
            {r['extra']}
            AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{window_start_utc}')
            AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{window_end_utc}')
          GROUP BY 1,2,3,4
          ORDER BY Date, Hour, PAYLOAD_BONUSCODE
        """

        df_r = spark.sql(query).toPandas()
        df_r['Week'] = week_val
        df_r['Date Time Range'] = date_time_range
        df_r['Report Name'] = r['name']

        if len(df_r) == 0:
            df_r = pd.DataFrame(
                [['', '', '', '(no data this window)', '', '', '', week_val, date_time_range, r['name']]],
                columns=COLUMNS
            )
        else:
            df_r = df_r[COLUMNS]

        values = df_r.astype(str).values.tolist()

        ws.append_rows(values, value_input_option='USER_ENTERED', table_range='A1')
        total_rows_pushed += len(df_r)
        print(f"[{r['name']}] appended {len(df_r)} rows")

    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        print(f"[{r['name']}] FAILED — {error_msg}")
        failures.append({"report": r['name'], "error": error_msg, "window": date_time_range})
        continue

print(f"\n--- Run summary ---")
print(f"Window: {date_time_range}")
print(f"Total rows pushed: {total_rows_pushed}")
print(f"Reports failed: {len(failures)} / {len(reports)}")

In [0]:
if failures:
    try:
        fail_ws = sh.worksheet("Failures")
        fail_existing = fail_ws.col_values(1)
        fail_next_row = len(fail_existing) + 1
    except gspread.exceptions.WorksheetNotFound:
        fail_ws = sh.add_worksheet(title="Failures", rows=1000, cols=4)
        fail_ws.update('A1', [['Window', 'Report', 'Error']])
        fail_next_row = 2

    fail_rows = [[f['window'], f['report'], f['error']] for f in failures]
    fail_ws.update(f'A{fail_next_row}', fail_rows, value_input_option='USER_ENTERED')

    raise Exception(f"{len(failures)} report(s) failed this run: {[f['report'] for f in failures]}")